# Rainfall Forecasting

## Project Overview

Forecasts **daily rainfall** (mm) over a 14-day horizon. Synthetic data spans 2 years with seasonal wet/dry patterns, intermittent rain events, and climate variability.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily rainfall data, predict the next 14 days. Rainfall forecasts are critical for agriculture, flood management, water resource planning, and urban drainage systems.

## Why This Project Matters

Rainfall drives agricultural productivity, flood risk, and water supply. Too little causes drought and crop failure; too much causes flooding and infrastructure damage. Accurate forecasts enable planting decisions, reservoir management, and flood early warnings.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily rainfall
- Seasonal pattern (wet season vs dry season)
- Intermittent — many dry days with occasional rain events
- Multi-day storm events
- No weekly pattern

| Column | Description |
|--------|-------------|
| `date` | Date |
| `rainfall_mm` | Daily rainfall (mm) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'rainfall_mm'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Seasonal wet/dry probability
wet_prob = 0.3 + 0.2 * np.sin(2 * np.pi * (t - 60) / 365)  # spring/summer wetter

# Generate rainfall with intermittency
rainfall_mm = np.zeros(N_POINTS)
in_storm = False
storm_duration = 0
for i in range(N_POINTS):
    if in_storm and storm_duration > 0:
        rainfall_mm[i] = np.random.exponential(8)
        storm_duration -= 1
        if storm_duration == 0:
            in_storm = False
    elif np.random.random() < wet_prob[i]:
        rainfall_mm[i] = np.random.exponential(5)
        if np.random.random() < 0.3:  # multi-day storm
            in_storm = True
            storm_duration = np.random.randint(1, 4)
    else:
        rainfall_mm[i] = 0

# Ensure positive and add minimum baseline
rainfall_mm = np.maximum(rainfall_mm, 0.1).round(1)

df = pd.DataFrame({'date': dates, 'rainfall_mm': rainfall_mm})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,rainfall_mm
0,2022-01-01,0.1
1,2022-01-02,0.1
2,2022-01-03,0.1
3,2022-01-04,0.1
4,2022-01-05,0.1


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('rainfall_mm Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("rainfall_mm Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

rainfall_mm Statistics:
count    730.000000
mean       2.282192
std        4.764784
min        0.100000
25%        0.100000
50%        0.100000
75%        2.475000
max       58.800000
Name: rainfall_mm, dtype: float64

CV: 2.088
Skewness: 4.286


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        1.2 | RMSE:        4.6 | MAPE:  7.10%
Seasonal Naive (7)             | MAE:        2.5 | RMSE:        6.5 | MAPE: 130.67%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared   R-Squared      RMSE  Time Taken
Model                                                                           
RANSACRegressor                    1695.000000 -129.307692  7.643672    0.065654
AdaBoostRegressor                   722.654233  -54.511864  4.988959    0.078333
PassiveAggressiveRegressor          303.462301  -22.266331  3.229839    0.006846
ExtraTreeRegressor                  157.671598  -11.051661  2.324558    0.007447
DecisionTreeRegressor               157.671598  -11.051661  2.324558    0.011802
DummyRegressor                      136.453093   -9.419469  2.161423    0.006302
Lasso                               134.649438   -9.280726  2.146984    0.008171
LassoLars                           134.649438   -9.280726  2.146984    0.007294
ElasticNet                          105.531782   -7.040906  1.898757    0.007687
OrthogonalMatchingPursuit            90.123822   -5.855679  1.753243    0.006629


## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: catboost
FLAML (catboost)               | MAE:        0.9 | RMSE:        1.0 | MAPE: 85.21%
FLAML Test (catboost)          | MAE:        2.4 | RMSE:        4.6 | MAPE: 128.19%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        3.1 | RMSE:        4.5 | MAPE: 206.30%
SF AutoETS                     | MAE:        2.7 | RMSE:        4.5 | MAPE: 166.86%
SF SeasonalNaive               | MAE:        1.2 | RMSE:        4.6 | MAPE:  7.10%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
                model      MAE     RMSE       MAPE
     FLAML (catboost) 0.932530 0.986851  85.209636
   Naive (Last Value) 1.235714 4.623619   7.101806
     SF SeasonalNaive 1.235714 4.623619   7.101806
FLAML Test (catboost) 2.396871 4.647812 128.193527
   Seasonal Naive (7) 2.471429 6.538785 130.673235
           SF AutoETS 2.709773 4.473241 166.856801
         SF AutoARIMA 3.079891 4.541259 206.296589

Top 3:
             model      MAE     RMSE      MAPE
  FLAML (catboost) 0.932530 0.986851 85.209636
Naive (Last Value) 1.235714 4.623619  7.101806
  SF SeasonalNaive 1.235714 4.623619  7.101806


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -0.03, Std: 4.65


## Interpretation and Business Insight

- Rainfall is inherently intermittent — many near-zero days with occasional events
- Seasonal pattern: spring/summer tends to be wetter than winter
- Multi-day storm events create clustered wet periods
- Rainfall is one of the hardest variables to forecast precisely
- Probabilistic forecasts are more useful than point forecasts for rainfall

## Limitations

1. Synthetic — real rainfall depends on terrain, atmospheric dynamics, ENSO
2. No atmospheric variables (pressure, humidity, wind)
3. Point location — real rainfall varies enormously over short distances
4. Daily totals miss intensity (drizzle vs downpour)
5. No distinction between convective and frontal rainfall

## How to Improve This Project

1. Add atmospheric variables from NWP models
2. Use radar/satellite data for nowcasting
3. Model rainfall occurrence and amount separately (hurdle model)
4. Include topographic features for spatial modeling
5. Add climate indices (ENSO, MJO) for seasonal forecasts

## Production Considerations

- Daily rainfall probability and amount forecasting
- Integration with flood early warning systems
- Agricultural irrigation scheduling
- Reservoir inflow prediction for water supply

## Common Mistakes

1. Using point forecasts for inherently probabilistic rainfall
2. Not handling intermittency (many zeros)
3. Evaluating with MAPE (undefined for zero values)
4. Treating all rainfall as equal (intensity matters)

## Mini Challenge / Exercises

1. Build a hurdle model: first predict wet/dry, then amount if wet
2. Create a 7-day rainfall total forecast and compare accuracy
3. Detect multi-day storm events in the data
4. Compare dry season vs wet season forecast accuracy

## Final Summary / Key Takeaways

1. Rainfall forecasting is inherently challenging due to intermittency
2. Seasonal patterns provide a baseline probability
3. Probabilistic forecasts are more useful than point predictions
4. Real deployment needs atmospheric data and radar inputs
5. Multi-day storm detection is the highest-value application

In [18]:
import json
metrics = {
    'project': 'Rainfall Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Rainfall Forecasting — Complete ===")

Metrics saved.

=== Rainfall Forecasting — Complete ===
